# 異常検知のための畳み込みオートエンコーダー
## 学習目標
1. ディープ畳み込みオートエンコーダーの構築方法を学ぶ
2. トレーニング済みオートエンコーダーを異常検知に使用する方法を学ぶ

このラボでは、オートエンコーダーモデルの構築方法を学びます。オートエンコーダーは、エンコーダーとデコーダーで構成される代表的な教師なし学習モデルです。<br>
オートエンコーダーは、入力を出力にコピーするようにトレーニングされた特殊なニューラルネットワークです。例えば、手書き数字の画像が与えられると、オートエンコーダーはまず画像をより低次元の潜在表現にエンコードし、次にその潜在表現を画像へとデコードします。オートエンコーダーは、再構成誤差（reconstruction error）を最小限に抑えながらデータを圧縮することを学習します。

オートエンコーダーには多くの活用法があります。このラボでは、オートエンコーダーがトレーニングされたデータ分布からの入力と比較して、異常な入力の再構成が困難であるという特性を活用して、異常検知にオートエンコーダーを使用します。

オートエンコーダーの詳細については、Ian Goodfellow、Yoshua Bengio、Aaron Courville 著『[Deep Learning](https://www.deeplearningbook.org/)』の第 14 章を参照してください。

In [ ]:
import os
import warnings

os.environ["KERAS_BACKEND"] = "tensorflow"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import keras

# ライブラリとモジュールのインポート
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

np.set_printoptions(threshold=np.inf)

### MNIST データの読み込み

手書き数字の白黒画像で構成される MNIST データセットを使用します。各画像は 28 x 28 ピクセルで、白と黒の間のグレーレベルを示す単一のチャンネルを持ち、shape(28, 28, 1) のテンソル表現を生成します。Keras にはこのデータセットがあらかじめ組み込まれており、次のセルで読み込みます。

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
mnist_digits = np.concatenate([x_train, x_test], axis=0)
mnist_digits = np.expand_dims(mnist_digits, -1).astype("float32") / 255

In [ ]:
plt.imshow(mnist_digits[0], cmap="Greys_r")

## オートエンコーダーモデルの構築
オートエンコーダーモデルの構築を始めましょう。 

オートエンコーダーは自己教師あり学習（self-supervised）モデルであり、入力とラベルに同じデータを使用します。 

![image](https://user-images.githubusercontent.com/6895245/164359926-f72472ca-f2de-4098-bf2b-3e649749f721.png)
入力の次元を固定サイズの潜在空間（エンコーダー）に圧縮し、その埋め込みから入力を「再構成」（デコーダー）しようとします。そのため、エンコーダー部分は PCA のような次元削減モデルと見なすことができます。ただし、PCA が入力を低次元空間に射影する線形変換を学習するのに対し、オートエンコーダーはニューラルネットワークを使用して非線形変換を学習します。

### エンコーダー
まずエンコーダー部分を構築しましょう。エンコーダーは入力から潜在空間へと変換します。
![image](https://user-images.githubusercontent.com/6895245/164373070-f7860451-1720-4e4b-aa91-460c4aa98020.png)

In [ ]:
def build_encoder(
    input_shape,
    latent_dim,
    filters_1=32,
    filters_2=64,
    kernel_size_1=3,
    kernel_size_2=3,
    strides_1=2,
    strides_2=2,
):
    encoder = keras.Sequential(
        [
            keras.Input(shape=input_shape, name="image_input"),
            layers.Conv2D(
                filters=filters_1,
                kernel_size=kernel_size_1,
                activation="relu",
                strides=strides_1,
                padding="same",
                name="encoder_convolution_1",
            ),
            layers.Conv2D(
                filters=filters_2,
                kernel_size=kernel_size_2,
                activation="relu",
                strides=strides_2,
                padding="same",
                name="encoder_convolution_2",
            ),
            layers.Flatten(name="encoder_flatten"),
            layers.Dense(latent_dim, name="encoder_output"),
        ],
        name="encoder",
    )
    return encoder

In [ ]:
LATENT_DIM = 2  # 可視化を容易にするため
INPUT_SHAPE = (28, 28, 1)  # 入力データのサイズ

encoder = build_encoder(INPUT_SHAPE, LATENT_DIM)
encoder.summary()

### デコーダー
デコーダーは、潜在空間から元の再構成画像へと戻します。
元の形状 (28, 28, 1) を再構成するために、[`Conv3DTranspose` レイヤー](https://keras.io/api/layers/convolution_layers/convolution3d_transpose/)（別名: 逆畳み込み / Deconvolution）を使用します。

![image](https://user-images.githubusercontent.com/6895245/164372979-29797db4-d108-4955-8056-ec668465c3f4.png)

In [ ]:
def build_decoder(latent_dim, encoder):
    decoder = keras.Sequential(
        [
            keras.Input(shape=(latent_dim,), name="decoder_input"),
            layers.Dense(
                encoder.layers[-1].input.shape[1],  # 3136
                activation="relu",
                name="decoder_dense",
            ),
            layers.Reshape(
                encoder.layers[-2].input.shape[1:],  # ([7, 7, 64])
                name="decoder_reshape",
            ),
            layers.Conv2DTranspose(
                filters=encoder.layers[-3].input.shape[-1],  # 32
                kernel_size=encoder.layers[-3].kernel_size,  # 3
                activation="relu",
                strides=encoder.layers[-3].strides,  # 2
                padding="same",
                name="decoder_deconvolution_1",
            ),
            layers.Conv2DTranspose(
                filters=encoder.layers[-4].input.shape[-1],  # 1
                kernel_size=encoder.layers[-4].kernel_size,  # 3
                activation="sigmoid",
                strides=encoder.layers[-4].strides,  # 2
                padding="same",
                name="decoder_deconvolution_2",
            ),
        ],
        name="decoder",
    )
    return decoder

In [ ]:
decoder = build_decoder(LATENT_DIM, encoder)
decoder.summary()

### オートエンコーダー全体の構築

オートエンコーダーはエンコーダーブロックとデコーダーブロックで構成されます。これら 2 つをシンプルにスタックして、オートエンコーダーモデル全体を構築しましょう。

In [ ]:
def build_autoencoder(input_shape, latent_dim):
    encoder = build_encoder(input_shape, latent_dim)
    decoder = build_decoder(latent_dim, encoder)
    autoencoder = keras.Sequential([encoder, decoder])
    autoencoder.build(input_shape=(None, *input_shape))
    return encoder, decoder, autoencoder

In [ ]:
encoder, decoder, autoencoder = build_autoencoder(INPUT_SHAPE, LATENT_DIM)

autoencoder.summary()

### 再構成損失（Reconstruction Loss）

オートエンコーダーの最適化目標は、入力と出力の間の再構成誤差を最小化することです。そこで、損失関数として MSE を選択します。

In [ ]:
autoencoder.compile(optimizer=keras.optimizers.Adam(), loss="mse")

## オートエンコーダーのトレーニング
`.fit()` を呼び出してトレーニングを実行しましょう。
ここで、特徴量とラベルの両方が同じ形状、つまり (28, 28, 1) を持っていることに注意してください。

In [ ]:
history = autoencoder.fit(mnist_digits, mnist_digits, epochs=30, batch_size=128)

再構成損失が徐々に減少していることが確認できます。

## 潜在空間の可視化

`LATENT_DIM=2` でオートエンコーダーモデルをトレーニングしました。2 次元の潜在空間（エンコーダーの出力）を可視化し、入力データからどのようにパターンを見出したかを確認してみましょう。

以下の関数は、入力画像をエンコーダーに渡すことで潜在空間にプロットします。また、実際の画像ラベル（オートエンコーダーのトレーニング中には使用されなかったもの）の表示も追加します。オートエンコーダーが同じラベルを持つ画像を、潜在空間内の比較的明確に定義された点のクラスターにグループ化できていることが観察できます。ある意味で、オートエンコーダーはトレーニング中にラベルの知識を持たずに、生の画像から画像ラベルを「推測」したと言えます！

In [ ]:
def plot_label_clusters(encoder, data, labels):
    # 潜在空間における数字クラスの 2D プロットを表示
    z_mean = encoder.predict(data)
    plt.figure(figsize=(12, 10))
    plt.scatter(z_mean[:, 0], z_mean[:, 1], c=labels)
    plt.colorbar()
    for label in range(10):  # MNIST、10 個の数字
        cx = np.mean(z_mean[labels == label, 0])
        cy = np.mean(z_mean[labels == label, 1])
        plt.text(
            cx, cy, str(label), color="white", fontsize=25, fontweight="bold"
        )
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.show()

In [ ]:
(x_train, y_train), _ = keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train, -1).astype("float32") / 255
plot_label_clusters(encoder, x_train, y_train)

数字のクラスターが確認できます！これは、エンコーダー部分が何らかのパターンを見つけ、この 2 次元の潜在空間に情報を埋め込んだことを意味します。

## デコーダーの可視化
次に、デコーダー部分も見てみましょう。
以下のヘルパー関数は、指定された範囲で 2 次元潜在空間のグリッドペア（`grid_x` と `grid_y`）を作成し、それらをデコーダーに渡してデコーダーモデルの出力をプロットします。<br>
これにより、デコーダーが潜在空間から出力画像をどのように生成するかを理解するのに役立ちます。

上記に示されたエンコーダー出力の潜在空間におけるクラスターの実際の範囲に基づいて、`[x_min, x_max, y_min, y_max]` 形式で `ranges` 引数を変更できます。

In [ ]:
def plot_latent_space(decoder, n=30, ranges=[-1, 1, -1, 1], figsize=15):
    # 数字の n*n 2D 多様体（manifold）を表示
    digit_size = 28
    figure = np.zeros((digit_size * n, digit_size * n))
    # 潜在空間における数字クラスの 2D プロットに対応する
    # 等間隔の座標
    grid_x = np.linspace(ranges[0], ranges[1], n)
    grid_y = np.linspace(ranges[2], ranges[3], n)[::-1]

    samples = [[x, y] for y in grid_y for x in grid_x]
    x_decoded = decoder.predict(np.array(samples))
    figure = np.zeros((digit_size * n, digit_size * n))
    for i in range(n):
        for j in range(n):
            figure[
                i * digit_size : (i + 1) * digit_size,
                j * digit_size : (j + 1) * digit_size,
            ] = x_decoded[i * n + j].reshape(digit_size, digit_size)

    plt.figure(figsize=(figsize, figsize))
    start_range = digit_size // 2
    end_range = n * digit_size + start_range
    pixel_range = np.arange(start_range, end_range, digit_size)
    sample_range_x = np.round(grid_x, 1)
    sample_range_y = np.round(grid_y, 1)
    plt.xticks(pixel_range, sample_range_x)
    plt.yticks(pixel_range, sample_range_y)
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.imshow(figure, cmap="Greys_r")
    plt.show()

In [ ]:
plot_latent_space(decoder, n=30, ranges=[-10, 10, -3, 10], figsize=15)

うまく機能しているようです！オートエンコーダーモデルは、2 次元の潜在空間から画像を正常に再構成しています。

次に、異常検知問題を解決するためにオートエンコーダーをどのように使用できるかを見てみましょう。

## 異常検知
オートエンコーダーの仕組みを実証できたので、このモデルを異常検知に活用してみましょう。

しかし、なぜこれが異常検知に有効なのでしょうか？

オートエンコーダーは、情報を圧縮して再構成することでデータのパターンを学習します。そのため、「正常」なデータで適切にトレーニングされていれば、これまで見てきたように「正常」な入力に対して類似した画像を再構築できるはずです。しかし、入力が「異常」であった場合、異常データのパターンは正常データとは大きく異なるため、類似した画像を再構成することはできません。

したがって、異常なケースでは、入力と出力の間の誤差が正常なケースよりも大きくなるはずです。オートエンコーダーのこの特性を異常検知問題に活用できます。

### 異常検知用オートエンコーダーの構築

実際に異常検知にオートエンコーダーを使用する前に、より大きな潜在次元を指定して、より高性能なモデルを作成しましょう。

潜在次元が大きいほど、モデルはより広い空間により豊かな情報を埋め込むことができます。

In [ ]:
encoder, decoder, autoencoder = build_autoencoder(INPUT_SHAPE, latent_dim=32)

In [ ]:
autoencoder.compile(optimizer=keras.optimizers.Adam(), loss="mse")
history = autoencoder.fit(mnist_digits, mnist_digits, epochs=30, batch_size=128)

### 異常の検知

次のセルでは、指定された入力の再構成誤差を計算し、それがしきい値を超えているかどうかをチェックすることで異常を検知します。しきい値は、以下の式に従ってデータセット上の再構成誤差の平均と標準偏差を計算することによって設定します。

`threshold = mean(reconstruction_error) + std(reconstruction_error) + sigma`

ここで、sigma は設定する必要があるハイパーパラメータです。

In [ ]:
num_sample = 2000  # しきい値計算用のサンプル数

reconstructions = autoencoder.predict(x_train[:num_sample])

train_loss = keras.ops.mean(
    keras.losses.mean_absolute_error(reconstructions, x_train[:num_sample]),
    axis=(1, 2),
)

In [ ]:
# 損失値のヒストグラムを可視化
plt.hist(train_loss, bins=50)
plt.xlabel("Train loss")
plt.ylabel("No of examples")
plt.show()

平均より標準偏差 2 つ分大きいしきい値を選択しましょう。

In [ ]:
# 異常しきい値の定義
sigma = 2

threshold = np.mean(train_loss) + np.std(train_loss) * sigma
print("Threshold: ", threshold)

In [ ]:
# 損失値のヒストグラムと異常しきい値を可視化
fig = plt.figure()
plt.hist(train_loss, bins=50)
plt.xlabel("Train loss")
plt.ylabel("No of examples")
plt.vlines(threshold, 0, 160, color="red")
plt.show()

### 予測
予測時には以下を行います。

1. オートエンコーダーを使用して入力を再構成する
1. その入力に対する再構成誤差を計算する
1. 再構成誤差が設定したしきい値を超えている場合、入力を異常として分類する

In [ ]:
def predict(model, data, threshold):
    reconstructions = model(data)
    loss = keras.ops.mean(
        keras.losses.mean_absolute_error(reconstructions, data), axis=(1, 2)
    )
    return keras.ops.greater(loss, threshold), loss[0], reconstructions

以下のヘルパー関数は、入力、再構成された出力、および実際の損失値としきい値の比較による異常検知の結果を可視化します。

In [ ]:
def plot_reconstruction(model, data, threshold):
    is_anomaly, loss, reconstruction = predict(
        model, keras.ops.expand_dims(data, 0), threshold
    )

    f, axarr = plt.subplots(1, 2)
    lossMSG = f"loss:{loss:.4f}"
    thMSG = f"threshold:{threshold:.4f}"
    text_anomaly = (
        f"Anomaly ({lossMSG} > {thMSG})"
        if is_anomaly
        else f"Normal ({lossMSG} <= {thMSG})"
    )
    title = f.suptitle(text_anomaly)
    c = "r" if is_anomaly else "g"
    plt.setp(title, color=c)

    axarr[0].imshow(data, cmap="Greys_r")
    axarr[0].set_title("Original Image")
    axarr[1].imshow(reconstruction[0], cmap="Greys_r")
    axarr[1].set_title("Reconstructed Image")
    f.show()

### 正常データセット
データセットからいくつかの正常データを渡して、オートエンコーダーモデルがどのように応答するかを見てみましょう。

In [ ]:
for d in x_train[num_sample - 10 : num_sample]:
    plot_reconstruction(autoencoder, d, threshold)

オートエンコーダーは画像を非常によく再構成しており、ほとんどのケースで損失値がしきい値未満になっているようです。<br>
これは私たちが期待していた通りの結果です！

### 異常データのテスト
では、「通常と異なる」データを受け取った場合はどうなるでしょうか？以下でいくつかの摂動（ノイズや変換）を作成し、何が起こるかを確認してみましょう。次のデータを試します。
- 90度回転した画像
- ネガ反転した画像
- ホワイトノイズを加えた画像

#### 90度回転した画像

In [ ]:
for d in x_train[num_sample - 10 : num_sample]:
    plot_reconstruction(autoencoder, np.rot90(d), threshold)

回転した画像はトレーニングやしきい値の計算に含まれていないため、回転した画像の多くが異常として検知されているようです。

ただし、`0` という値は角度に対して不変であるため、回転した `0` は正常として検知されることがあります。

次に、元の画像からネガ画像を作成し、モデルに入力するとどうなるかを見てみましょう。

#### ネガ反転した画像

In [ ]:
for d in x_train[num_sample - 10 : num_sample]:
    plot_reconstruction(autoencoder, 1 - d, threshold)

これらはかなりの損失値とともに異常として検知されています！そして、モデルが元の画像をまったく再構成できていないことがわかります。

では、ノイズの多いデータはどうでしょうか？

#### ホワイトノイズを加えた画像

In [ ]:
def add_noise_and_scale(d):
    noise = np.random.normal(0, 0.05, size=(28, 28, 1))
    _d = d + noise
    return _d - np.min(_d) / (np.max(_d) - np.min(_d))


for d in x_train[num_sample - 10 : num_sample]:
    plot_reconstruction(autoencoder, add_noise_and_scale(d), threshold)

これらも異常として検知されました！

興味深いことに、オートエンコーダーはクリーンな画像でトレーニングされているため、「ノイズを除去（デノイズ）」しようとして、きれいな画像を出力として生成します。その結果、入力ときれいな出力の間の誤差が大きくなり、異常として検知されます。

これは、オートエンコーダーをノイズ除去の目的にも活用できることを意味しているのでしょうか？はい、[ノイズ除去オートエンコーダー（Denoising Autoencoders）](https://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf)を使用することで実現できます。<br>
ただし、その場合はオートエンコーダーにクリーンなデータのパターンを捉えさせるために、トレーニングデータにクリーンな画像を使用することを忘れないでください。


## まとめ
このラボでは、エンコーダーとデコーダーをスタックしてオートエンコーダーモデルを構築する方法を学びました。<br>
また、異常検知の目的でオートエンコーダーを活用する方法も学びました。

# License

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.